# 60 — Validate the BC A-Box against the published shapes

The A-Box **does not conform**, and that is the deliverable of this notebook
rather than a problem with it (decision D11).

Every violation traces to one of four causes, each a place where the published
model describes a document and the rendering describes a graph. The notebook
classifies every result and **fails if a violation appears that is not one of
them** — so an unexplained finding cannot hide inside a large count.

Requires `pyshacl`.

## Configuration

In [ ]:
ROOT    = ".."
REPORTS = "../reports"

INSTANCES = f"{ROOT}/cosmos_bc_v1.instances.ttl"
SHAPES    = f"{ROOT}/cosmos_bc_v1.shapes.ttl"

BC_NS = "https://www.cdisc.org/cosmos/biomedical_concept_v1.0/"

# Every expected (constraint, path) pair, and why it happens.
# A result outside this map fails the notebook.
EXPECTED = {
    ("ClosedConstraintComponent", "identifier"):
        "authored: dcterms:identifier carries the bare C-code (D2)",
    ("ClosedConstraintComponent", "exactMatch"):
        "authored: skos:exactMatch to the EVS form (D2, usdm-rdf D4)",
    ("InConstraintComponent", "packageType"):
        "generator disagreement: gen-owl declares enum values as class IRIs, gen-shacl expects strings",
    ("InConstraintComponent", "resultScales"):
        "generator disagreement: gen-owl declares enum values as class IRIs, gen-shacl expects strings",
    ("InConstraintComponent", "dataType"):
        "generator disagreement: gen-owl declares enum values as class IRIs, gen-shacl expects strings",
    ("DatatypeConstraintComponent", "parentConceptId"):
        "published range is string; the A-Box renders the parent reference as a link",
    ("NodeKindConstraintComponent", "parentConceptId"):
        "published range is string; the A-Box renders the parent reference as a link",
    ("MaxCountConstraintComponent", "shortName"):
        "dual-role concept whose BC and DEC labels differ (D12)",
}

# Closed-shape violations on a concept's own slots arise only where one node
# carries both types; the focus node must be one of the dual-role concepts.
DUAL_ROLE_CLOSED = {
    "categories", "exampleSet", "dataElementConcepts", "synonyms",
    "packageType", "definition", "packageDate", "dataType", "resultScales",
    "parentConceptId", "shortName", "ncitCode", "coding", "system", "systemName", "code",
}

## Validate

In [ ]:
import csv
from collections import Counter
from pathlib import Path

from pyshacl import validate
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, SH

data = Graph().parse(INSTANCES, format="turtle")
shapes = Graph().parse(SHAPES, format="turtle")

conforms, results, _ = validate(data, shacl_graph=shapes, inference="none", advanced=True)
print(f"conforms: {conforms}")
print(f"results:  {len(list(results.subjects(RDF.type, SH.ValidationResult))):,}")

## Classify every result

`local(path)` reduces a predicate IRI to its last segment, which is enough to
group by — the shapes and the data share one vocabulary.

In [ ]:
def local(term):
    if term is None:
        return "-"
    text = str(term)
    return text.rsplit("#", 1)[-1].rsplit("/", 1)[-1]


dual_role_codes = set()
report = Path(REPORTS, "dual_role_concepts.csv")
if report.exists():
    with open(report, encoding="utf-8") as f:
        dual_role_codes = {row["ncit_code"] for row in csv.DictReader(f)}

rows, counts, unexplained = [], Counter(), []

for result in results.subjects(RDF.type, SH.ValidationResult):
    component = local(results.value(result, SH.sourceConstraintComponent))
    path = local(results.value(result, SH.resultPath))
    focus = str(results.value(result, SH.focusNode))
    code_ = focus.rsplit("_", 1)[-1] if "NCIT_" in focus else ""

    cause = EXPECTED.get((component, path))
    if cause is None and component == "ClosedConstraintComponent" and path in DUAL_ROLE_CLOSED:
        if code_ in dual_role_codes:
            cause = "dual-role concept: one node typed as both BC and DEC, each closed shape rejects the other's slots (D12)"

    if cause is None:
        unexplained.append((component, path, focus))
    else:
        counts[(component, path, cause)] += 1

    rows.append({
        "constraint": component,
        "path": path,
        "focus_node": focus,
        "cause": cause or "UNEXPLAINED",
    })

for (component, path, cause), n in counts.most_common():
    print(f"{n:>6,}  {component:30s} {path:18s} {cause[:70]}")
print()
print(f"unexplained results: {len(unexplained):,}")
for u in unexplained[:10]:
    print("   ", u)

## Write the conformance report

In [ ]:
out = Path(REPORTS, "shacl_conformance.csv")
with open(out, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["constraint", "path", "focus_node", "cause"])
    writer.writeheader()
    writer.writerows(sorted(rows, key=lambda r: (r["cause"], r["constraint"], r["path"], r["focus_node"])))

summary = Path(REPORTS, "shacl_conformance_summary.csv")
with open(summary, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["constraint", "path", "cause", "count"])
    for (component, path, cause), n in sorted(counts.items(), key=lambda kv: -kv[1]):
        writer.writerow([component, path, cause, n])

print(f"wrote {out}       ({len(rows):,} rows)")
print(f"wrote {summary}   ({len(counts)} causes)")

## Result

Non-conformance is expected and recorded. What is **not** acceptable is a
violation nobody has accounted for.

In [ ]:
if unexplained:
    raise RuntimeError(
        f"{len(unexplained):,} validation result(s) fall outside the causes decision D11 records. "
        "Read reports/shacl_conformance.csv before changing anything - an unexplained violation "
        "is either a rendering bug or a new finding, and both matter."
    )

print(f"{len(rows):,} violations, all four causes accounted for.")
print()
print("The A-Box does not conform to the published shapes, by design:")
print("  - two causes are this repo adding identity the schema has no slot for")
print("  - one is gen-owl and gen-shacl disagreeing about what an enum value is")
print("  - one is a reference the published model types as a string")